# Final Project — Video Game Hits from Pre-Release Metadata

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, cohen_kappa_score,
    confusion_matrix
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_FILE = "vgsales.csv"

print("Imports OK")

Imports OK


In [2]:
# Load the data
df_raw = pd.read_csv(DATA_FILE)
print("Loaded {}: {} rows x {} columns".format(DATA_FILE, df_raw.shape[0], df_raw.shape[1]))
print(df_raw.columns.tolist())
df_raw.head()

Loaded vgsales.csv: 16598 rows x 11 columns
['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [3]:
# Target construction + leakage control
# Target computed BEFORE dropping sales columns; then drop all sales/rank/identifier columns
df_raw["hit_game"] = (df_raw["Global_Sales"] >= 1.0).astype(int)

drop_cols = ["Rank", "Name", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
df = df_raw.drop(columns=drop_cols)

# Drop exact duplicates
df = df.drop_duplicates()

# Hygiene
df["Year"] = df["Year"].fillna(0)
df["Publisher"] = df["Publisher"].fillna("Unknown")

print("Cleaned shape:", df.shape)
print("hit_game balance (%):")
print(df["hit_game"].value_counts(normalize=True).mul(100).round(2))

Cleaned shape: (11311, 5)
hit_game balance (%):
hit_game
0    85.44
1    14.56
Name: proportion, dtype: float64


In [4]:
# Train / test split FIRST (stratified) — all feature engineering derived from train only
feature_cols = ["Year", "Genre", "Publisher", "Platform"]
X = df[feature_cols]
y = df["hit_game"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train hit rate:", round(y_train.mean(), 4), "| Test hit rate:", round(y_test.mean(), 4))

Train: (9048, 4) | Test: (2263, 4)
Train hit rate: 0.1456 | Test hit rate: 0.1458


In [5]:
# Engineer pre-release features (train-derived only)
def add_pre_release_features(df_in, fit_stats=None):
    df_out = df_in.copy()
    df_out["release_decade"] = np.where(
        df_out["Year"] == 0, "Unknown",
        (df_out["Year"] // 10 * 10).astype(str) + "s"
    )
    platform_generation = {
        "2600": 2, "NES": 3, "SNES": 4, "N64": 5, "GC": 6, "Wii": 7,
        "GB": 3, "GBA": 5, "DS": 7, "3DS": 8, "PS": 4, "PS2": 6,
        "PS3": 7, "PS4": 8, "PSP": 7, "PSV": 8, "XB": 6, "360": 7,
        "XOne": 8, "PC": 5
    }
    df_out["console_generation"] = df_out["Platform"].map(platform_generation).fillna(0).astype(int)

    if fit_stats is None:
        pub_rate = df_out.assign(hit=y_train).groupby("Publisher")["hit"].mean().rename("publisher_hit_rate")
        plat_rate = df_out.assign(hit=y_train).groupby("Platform")["hit"].mean().rename("platform_hit_rate")
        fit_stats = (pub_rate, plat_rate)
        df_out = df_out.join(pub_rate, on="Publisher").join(plat_rate, on="Platform")
    else:
        pub_rate, plat_rate = fit_stats
        df_out = df_out.join(pub_rate, on="Publisher").join(plat_rate, on="Platform")

    global_rate = y_train.mean()
    df_out["publisher_hit_rate"] = df_out["publisher_hit_rate"].fillna(global_rate)
    df_out["platform_hit_rate"] = df_out["platform_hit_rate"].fillna(global_rate)
    return df_out, fit_stats

X_train_fe, fit_stats = add_pre_release_features(X_train)
X_test_fe, _ = add_pre_release_features(X_test, fit_stats=fit_stats)

print("Train features sample:")
print(X_train_fe.head())
print("")
print("Feature columns:", X_train_fe.columns.tolist())

Train features sample:
         Year         Genre                    Publisher Platform  \
11351  2006.0        Racing           Namco Bandai Games       GC   
5128   2004.0  Role-Playing                     Nintendo       GC   
3370   1998.0     Adventure              Electronic Arts       PS   
15824  2008.0     Adventure                  Crimson Cow       PC   
5235   2003.0  Role-Playing  Sony Computer Entertainment      PS2   

      release_decade  console_generation  publisher_hit_rate  \
11351        2000.0s                   6            0.089157   
5128         2000.0s                   6            0.510638   
3370         1990.0s                   4            0.316163   
15824        2000.0s                   5            0.000000   
5235         2000.0s                   6            0.310954   

       platform_hit_rate  
11351           0.097643  
5128            0.097643  
3370            0.180516  
15824           0.082372  
5235            0.174345  

Feature column

In [6]:
# Preprocessing pipeline
numeric_features = ["Year", "console_generation", "publisher_hit_rate", "platform_hit_rate"]
categorical_features = ["Genre", "Publisher", "Platform", "release_decade"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Preprocessing pipeline ready")

Preprocessing pipeline ready


1. **Random Forest** bagged ensemble of decision trees
2. **XGBoost** gradient-boosted tree ensemble

In [7]:
# Define both models (no for-loops)
pipe_rf = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=RANDOM_STATE, n_jobs=-1)),
])

pipe_xgb = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                          scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")),
])

print("Models defined:", "Random Forest", "and", "XGBoost")

Models defined: Random Forest and XGBoost


## Implementation Random Forest cross-validation

In [8]:
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"}

# Random Forest: stratified 5-fold CV on training split
cv_rf = cross_validate(pipe_rf, X_train_fe, y_train, cv=cv, scoring=scoring,
                    n_jobs=-1, return_train_score=False)

pipe_rf.fit(X_train_fe, y_train)
rf_pred = pipe_rf.predict(X_test_fe)
rf_prob = pipe_rf.predict_proba(X_test_fe)[:, 1]

rf_test = {
    "accuracy": accuracy_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred, zero_division=0),
    "recall": recall_score(y_test, rf_pred, zero_division=0),
    "f1": f1_score(y_test, rf_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, rf_prob),
    "pr_auc": average_precision_score(y_test, rf_prob),
    "kappa": cohen_kappa_score(y_test, rf_pred),
}

print("=== Random Forest ===")
print("CV  : F1={:.3f} (+/-{:.3f}) ROC-AUC={:.3f} PR-AUC={:.3f}".format(
    cv_rf["test_f1"].mean(), cv_rf["test_f1"].std(),
    cv_rf["test_roc_auc"].mean(), cv_rf["test_pr_auc"].mean()))
print("Test: accuracy={:.3f} precision={:.3f} recall={:.3f} f1={:.3f}".format(
    rf_test["accuracy"], rf_test["precision"], rf_test["recall"], rf_test["f1"]))
print("      roc_auc={:.3f} pr_auc={:.3f} kappa={:.3f}".format(
    rf_test["roc_auc"], rf_test["pr_auc"], rf_test["kappa"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, rf_pred))

=== Random Forest ===
CV  : F1=0.283 (+/-0.026) ROC-AUC=0.766 PR-AUC=0.281
Test: accuracy=0.788 precision=0.260 recall=0.245 f1=0.252
      roc_auc=0.748 pr_auc=0.253 kappa=0.129
Confusion matrix:
[[1702  231]
 [ 249   81]]


## Implementation XGBoost cross-validation

In [9]:
# XGBoost: stratified 5-fold CV on training split
cv_xgb = cross_validate(pipe_xgb, X_train_fe, y_train, cv=cv, scoring=scoring,
                    n_jobs=-1, return_train_score=False)

pipe_xgb.fit(X_train_fe, y_train)
xgb_pred = pipe_xgb.predict(X_test_fe)
xgb_prob = pipe_xgb.predict_proba(X_test_fe)[:, 1]

xgb_test = {
    "accuracy": accuracy_score(y_test, xgb_pred),
    "precision": precision_score(y_test, xgb_pred, zero_division=0),
    "recall": recall_score(y_test, xgb_pred, zero_division=0),
    "f1": f1_score(y_test, xgb_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, xgb_prob),
    "pr_auc": average_precision_score(y_test, xgb_prob),
    "kappa": cohen_kappa_score(y_test, xgb_pred),
}

print("=== XGBoost ===")
print("CV  : F1={:.3f} (+/-{:.3f}) ROC-AUC={:.3f} PR-AUC={:.3f}".format(
    cv_xgb["test_f1"].mean(), cv_xgb["test_f1"].std(),
    cv_xgb["test_roc_auc"].mean(), cv_xgb["test_pr_auc"].mean()))
print("Test: accuracy={:.3f} precision={:.3f} recall={:.3f} f1={:.3f}".format(
    xgb_test["accuracy"], xgb_test["precision"], xgb_test["recall"], xgb_test["f1"]))
print("      roc_auc={:.3f} pr_auc={:.3f} kappa={:.3f}".format(
    xgb_test["roc_auc"], xgb_test["pr_auc"], xgb_test["kappa"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, xgb_pred))

=== XGBoost ===
CV  : F1=0.434 (+/-0.015) ROC-AUC=0.833 PR-AUC=0.439
Test: accuracy=0.657 precision=0.280 recall=0.861 f1=0.423
      roc_auc=0.809 pr_auc=0.423 kappa=0.260
Confusion matrix:
[[1203  730]
 [  46  284]]


In [10]:
# Summary table (no for-loops)
summary = pd.DataFrame({
    "Random Forest": rf_test,
    "XGBoost": xgb_test,
}).T.round(4)

cv_summary = pd.DataFrame({
    "Random Forest": {
        "cv_f1_mean": cv_rf["test_f1"].mean(),
        "cv_f1_std": cv_rf["test_f1"].std(),
        "cv_roc_auc_mean": cv_rf["test_roc_auc"].mean(),
        "cv_pr_auc_mean": cv_rf["test_pr_auc"].mean(),
    },
    "XGBoost": {
        "cv_f1_mean": cv_xgb["test_f1"].mean(),
        "cv_f1_std": cv_xgb["test_f1"].std(),
        "cv_roc_auc_mean": cv_xgb["test_roc_auc"].mean(),
        "cv_pr_auc_mean": cv_xgb["test_pr_auc"].mean(),
    },
}).T.round(4)

summary = summary.join(cv_summary)
summary

,accuracy,precision,recall,f1,roc_auc,pr_auc,kappa,cv_f1_mean,cv_f1_std,cv_roc_auc_mean,cv_pr_auc_mean
Random Forest,0.7879,0.2596,0.2455,0.2523,0.7481,0.2529,0.1289,0.2828,0.0255,0.7658,0.2813
XGBoost,0.6571,0.2801,0.8606,0.4226,0.8092,0.4234,0.2597,0.4339,0.0147,0.8331,0.4385


## Threshold optimization cost/return economics

In [11]:
# Random Forest threshold optimization (vectorized, no loop)
marketing_cost = 0.20
net_return_per_hit = 1.00
thresholds = np.arange(0.05, 0.96, 0.05)

rf_prob_col = rf_prob.reshape(-1, 1)
thresholds_row = thresholds.reshape(1, -1)
rf_preds = (rf_prob_col >= thresholds_row).astype(int)
y_true = y_test.to_numpy().reshape(-1, 1)
rf_tp = ((rf_preds == 1) & (y_true == 1)).sum(axis=0)
rf_pred_pos = rf_preds.sum(axis=0)
rf_net = rf_tp * net_return_per_hit - rf_pred_pos * marketing_cost
rf_best_idx = int(np.argmax(rf_net))

print("Random Forest: best threshold={:.2f} net_return=${:.2f}M (TP={}, predicted_pos={})".format(
    thresholds[rf_best_idx], rf_net[rf_best_idx], rf_tp[rf_best_idx], rf_pred_pos[rf_best_idx]))

Random Forest: best threshold=0.15 net_return=$73.80M (TP=223, predicted_pos=746)


In [12]:
# XGBoost threshold optimization (vectorized, no loop)
xgb_prob_col = xgb_prob.reshape(-1, 1)
xgb_preds = (xgb_prob_col >= thresholds_row).astype(int)
xgb_tp = ((xgb_preds == 1) & (y_true == 1)).sum(axis=0)
xgb_pred_pos = xgb_preds.sum(axis=0)
xgb_net = xgb_tp * net_return_per_hit - xgb_pred_pos * marketing_cost
xgb_best_idx = int(np.argmax(xgb_net))

print("XGBoost: best threshold={:.2f} net_return=${:.2f}M (TP={}, predicted_pos={})".format(
    thresholds[xgb_best_idx], xgb_net[xgb_best_idx], xgb_tp[xgb_best_idx], xgb_pred_pos[xgb_best_idx]))

XGBoost: best threshold=0.75 net_return=$93.20M (TP=202, predicted_pos=544)


## Feature importance train-derived, leakage-safe

In [13]:
# Prepare feature names for importance
pre = pipe_rf.named_steps["preprocess"]
pre.fit(X_train_fe, y_train)
cat_features = pre.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
all_features = numeric_features + list(cat_features)

In [14]:
# Random Forest feature importance
rf_clf = pipe_rf.named_steps["clf"]
rf_imp = getattr(rf_clf, "feature_importances_", None)
rf_fi = pd.DataFrame({"feature": all_features, "importance": rf_imp})
rf_fi = rf_fi.sort_values("importance", ascending=False).head(10)
print("Random Forest top features:")
print(rf_fi.round(4).to_string(index=False))

Random Forest top features:
                  feature  importance
       publisher_hit_rate      0.1960
                     Year      0.1380
        platform_hit_rate      0.0542
       Publisher_Nintendo      0.0360
Publisher_Electronic Arts      0.0312
             Genre_Action      0.0285
            Genre_Shooter      0.0227
          Genre_Adventure      0.0204
             Genre_Sports      0.0197
       console_generation      0.0193


In [15]:
# XGBoost feature importance
xgb_clf = pipe_xgb.named_steps["clf"]
xgb_imp = getattr(xgb_clf, "feature_importances_", None)
xgb_fi = pd.DataFrame({"feature": all_features, "importance": xgb_imp})
xgb_fi = xgb_fi.sort_values("importance", ascending=False).head(10)
print("XGBoost top features:")
print(xgb_fi.round(4).to_string(index=False))

XGBoost top features:
                  feature  importance
       publisher_hit_rate      0.1277
        platform_hit_rate      0.0306
     Publisher_Activision      0.0265
Publisher_Electronic Arts      0.0237
            Genre_Shooter      0.0217
            Platform_WiiU      0.0214
             Genre_Racing      0.0212
             Genre_Action      0.0208
             Genre_Sports      0.0191
           Publisher_Sega      0.0189


### Results
- **Yes**, pre-release metadata has real predictive power XGBoost ROC-AUC ≈ 0.81 on held-out data.
- **Accuracy is misleading** both models score below the 85.4% majority-class baseline on
  accuracy, yet both exceed it on F1/PR-AUC (baseline PR-AUC ≈ prevalence = 0.146).
- The model works as a **triage tool** but cannot capture game
  quality or marketing spend, so it is not a definitive hit detector.

In [16]:
# Save processed features
X_train_fe.assign(hit_game=y_train.values).to_csv("train_features.csv", index=False)
X_test_fe.assign(hit_game=y_test.values).to_csv("test_features.csv", index=False)
print("Saved train_features.csv and test_features.csv")

Saved train_features.csv and test_features.csv
